<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# Group Relative Policy Optimization (GRPO)

In [3]:
# Group Relative Policy Optimization (GRPO) - Standalone Implementation
# Innovation: Group-based advantage estimation without value functions
# Formula: A_i = r_i - (1/N) ∑_{j=1}^N r_j (group mean baseline)

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import json
from datetime import datetime

# GRPO Configuration
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPERATURE = 0.8
MAX_LENGTH = 512
MAX_NEW_TOKENS = 200
LEARNING_RATE_GRPO = 2.5e-5
NUM_EPOCHS_GRPO = 8
BATCH_SIZE_GRPO = 1
GRAD_ACCUM_GRPO = 8
WARMUP_RATIO = 0.2
LOGGING_STEPS = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

TEST_QUESTIONS = [
    "How do I cook pasta perfectly?",
    "What's the best way to scramble eggs?",
    "How do I make rice that isn't sticky?",
    "What's an easy dinner for beginners?",
    "How do I know when chicken is cooked?",
]


def install_packages():
    packages = [
        "torch>=2.0.0",
        "transformers>=4.36.0",
        "trl>=0.7.4",
        "datasets>=2.14.0",
        "accelerate>=0.21.0",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        except:
            pass


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def monitor_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        return reserved < 7.5
    return True


def test_model(model, tokenizer, prompt):
    model.eval()
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            use_cache=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def evaluate_stage(model, tokenizer, stage_name):
    results = {}
    print(f"\n{'='*60}")
    print(f"{stage_name.upper()} MODEL EVALUATION")
    print(f"{'='*60}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\nQ{i}: {question}")
        print("-" * 40)
        response = test_model(model, tokenizer, question)
        results[question] = response
        print(f"Answer: {response}")

    return results


def create_interactive_cooking_scenarios():
    """Create challenging cooking scenarios for GRPO interactive training"""
    return [
        "I'm a complete beginner and want to make dinner for my family tonight. What should I cook that's foolproof but impressive?",
        "My cookies always spread too much and turn out flat and greasy. I've tried everything but they still fail. What's going wrong?",
        "I want to start meal prepping for the week but I'm overwhelmed and don't know where to begin. Help me create a simple system!",
        "Every time I try to make a sauce, it either breaks, curdles, or tastes bland. I'm ready to give up on sauces entirely.",
        "I'm trying to eat healthier but everything I cook tastes boring compared to takeout. How can I make healthy food actually taste good?",
    ]


def collect_human_feedback(model, tokenizer, prompt, num_responses=3):
    """Generate multiple responses and collect human ratings for GRPO training"""
    print(f"\n{'='*70}")
    print(f"GRPO INTERACTIVE FEEDBACK COLLECTION")
    print(f"{'='*70}")
    print(f"Prompt: {prompt}")
    print(f"Algorithm: Generate {num_responses} responses, collect human ratings")
    print(f"Baseline: Group mean eliminates need for value function")
    print(f"{'='*70}")

    responses = []
    model.eval()

    # Generate diverse responses with varied temperature for group sampling
    print("Generating diverse responses for group evaluation...")
    for i in range(num_responses):
        temp = TEMPERATURE + i * 0.3  # Increase diversity across group
        response = test_model(model, tokenizer, prompt)
        responses.append(response)
        print(f"Response {i+1} generated (temperature={temp:.1f})")

    # Collect human ratings for advantage calculation
    ratings = []
    print(f"\n{'='*70}")
    print("HUMAN EVALUATION INSTRUCTIONS")
    print("=" * 70)
    print("Rate each response on a 1-5 scale:")
    print("  1 = Poor (unhelpful, brief, discouraging)")
    print("  2 = Below Average (somewhat helpful)")
    print("  3 = Average (adequate information)")
    print("  4 = Good (helpful, detailed, encouraging)")
    print("  5 = Excellent (comprehensive, inspiring, practical)")
    print()
    print("Consider: helpfulness, detail level, encouragement, and practical value")
    print("=" * 70)

    for i, response in enumerate(responses):
        print(f"\n--- RESPONSE {i+1} ---")
        print("Content:")
        print(response)
        print("-" * 50)

        while True:
            try:
                rating_input = input(f"Rate Response {i+1} (1-5): ").strip()
                rating = int(rating_input)
                if 1 <= rating <= 5:
                    ratings.append((rating - 1) / 4.0)  # Normalize to [0,1]
                    print(f"Recorded: {rating}/5")
                    break
                else:
                    print("Please enter a number between 1 and 5")
            except ValueError:
                print("Please enter a valid number (1-5)")
            except KeyboardInterrupt:
                print("\nSkipping remaining ratings...")
                return responses[: len(ratings)], ratings

    return responses, ratings


def grpo_training_step(model, tokenizer, prompt, learning_rate=LEARNING_RATE_GRPO):
    """Single GRPO training step using policy gradients with group-based advantages"""
    print(f"\nGRPO TRAINING STEP")
    print(f"Algorithm: Policy gradients with group-relative advantages")
    print(f"Mathematical formula: ∇_θ J = E[∇_θ log π_θ(a|s) × A(s,a)]")
    print(f"Baseline strategy: Group mean eliminates value function requirement")

    # Collect group responses and human feedback
    responses, ratings = collect_human_feedback(model, tokenizer, prompt)

    if not ratings or len(ratings) == 0:
        print("No ratings collected, skipping training step.")
        return model

    # Calculate group-relative advantages (core GRPO innovation)
    group_baseline = sum(ratings) / len(ratings)  # Group mean baseline
    advantages = [
        (rating - group_baseline) for rating in ratings
    ]  # Relative advantages

    print(f"\n" + "=" * 60)
    print("GRPO ADVANTAGE ANALYSIS")
    print("=" * 60)
    print("Mathematical Foundation: A_i = r_i - (1/N) ∑_j r_j")
    print(f"Raw human ratings: {[round(r*4+1, 1) for r in ratings]} (1-5 scale)")
    print(f"Group baseline: {group_baseline:.3f}")
    print(f"Computed advantages: {[f'{a:+.3f}' for a in advantages]}")
    print()
    print("Interpretation:")
    print("  • Positive advantages → increase response probability")
    print("  • Negative advantages → decrease response probability")
    print("  • Zero advantages → no policy update")
    print("=" * 60)

    # GRPO policy gradient update
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    total_loss = 0
    updated_responses = 0

    print(f"\nExecuting policy gradient updates...")

    for i, (response, advantage) in enumerate(zip(responses, advantages)):
        # Skip responses with negligible advantage
        if abs(advantage) < 0.05:
            print(
                f"Response {i+1}: advantage={advantage:+.3f} (skipped - negligible impact)"
            )
            continue

        # Prepare training sequence
        full_text = f"{prompt}\n{response}"
        inputs = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

        if torch.cuda.is_available():
            inputs = {k: v.to(device) for k, v in inputs.items()}

        # Policy gradient loss weighted by advantage
        outputs = model(**inputs, labels=inputs["input_ids"])
        policy_loss = outputs.loss

        # GRPO loss: L = A(s,a) × cross_entropy_loss
        grpo_loss = advantage * policy_loss

        # Gradient step with clipping for stability
        optimizer.zero_grad()
        grpo_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += abs(grpo_loss.item())
        updated_responses += 1

        print(
            f"Response {i+1}: advantage={advantage:+.3f}, policy_loss={grpo_loss.item():.4f}"
        )

    print(f"\n" + "=" * 60)
    print("GRPO TRAINING STEP COMPLETED")
    print("=" * 60)
    print(f"Training Statistics:")
    print(f"  • Total loss: {total_loss:.4f}")
    print(f"  • Responses updated: {updated_responses}/{len(responses)}")
    print(f"  • Advantage range: [{min(advantages):.3f}, {max(advantages):.3f}]")
    print()
    print("Policy Updates:")
    print("  • High-rated responses: Increased generation probability")
    print("  • Low-rated responses: Decreased generation probability")
    print("  • Model learns from relative quality differences")
    print("=" * 60)

    return model


def run_grpo_training(model, tokenizer):
    """Interactive GRPO training with human feedback"""
    print("=" * 80)
    print("GROUP RELATIVE POLICY OPTIMIZATION (GRPO)")
    print("=" * 80)
    print("Mathematical Foundation: A_i = r_i - (1/N) ∑r_j")
    print("Training: Interactive human feedback with group baselines")
    print("Innovation: No value function needed, ~50% memory reduction vs PPO")

    cleanup_memory()

    scenarios = create_interactive_cooking_scenarios()
    print(f"\nInteractive Training: {len(scenarios)} scenarios")
    print("Process: Generate → Rate → Learn from your preferences")

    completed = 0
    max_sessions = min(3, len(scenarios))  # Limit for demonstration

    for i, scenario in enumerate(scenarios[:max_sessions]):
        print(f"\n{'='*20} SESSION {i+1}/{max_sessions} {'='*20}")

        try:
            model = grpo_training_step(model, tokenizer, scenario)
            completed += 1

            if i < max_sessions - 1:
                print("\n" + "-" * 50)
                continue_choice = (
                    input("Continue to next scenario? (y/n): ").strip().lower()
                )
                if continue_choice in ["n", "no"]:
                    print("Training stopped by user.")
                    break

        except KeyboardInterrupt:
            print("\n\nTraining interrupted by user.")
            break
        except Exception as e:
            print(f"Error in session: {e}")
            continue

    print(f"\nGRPO training complete: {completed}/{max_sessions} sessions")
    cleanup_memory()
    return model, tokenizer


def save_results_json(results, filename):
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def print_grpo_theory():
    """Print comprehensive GRPO theoretical foundation"""
    print("\n" + "=" * 80)
    print("GRPO THEORETICAL FOUNDATION")
    print("=" * 80)
    print("Mathematical Innovation: Group-relative advantages without value functions")
    print()
    print("Core Algorithm:")
    print("  Traditional PPO: ∇_θ J = E[∇_θ log π_θ(a|s) × (r - V_φ(s))]")
    print("  GRPO Innovation: ∇_θ J = E[∇_θ log π_θ(a|s) × (r_i - (1/N)∑r_j)]")
    print("  Key difference: Group mean baseline vs learned value function")
    print()
    print("Algorithmic Process:")
    print("  1. Generate group responses: {y_1, y_2, ..., y_N} ~ π_θ(·|x)")
    print("  2. Collect human ratings: r_i = human_evaluation(x, y_i)")
    print("  3. Calculate baseline: b = (1/N) ∑_{i=1}^N r_i")
    print("  4. Compute advantages: A_i = r_i - b")
    print("  5. Policy update: θ ← θ + α ∑_i A_i ∇_θ log π_θ(y_i|x)")
    print()
    print("Theoretical Advantages:")
    print("  • Memory Efficiency: ~50% reduction vs PPO (no critic network)")
    print("  • Training Stability: Group mean provides unbiased baseline")
    print("  • Architectural Simplicity: Single model vs actor-critic")
    print("  • Variance Reduction: Group sampling reduces gradient variance")
    print("  • Direct Learning: Human feedback directly guides optimization")
    print()
    print("Mathematical Justification:")
    print("  • Group mean approximates expected return: E[r(x,y)] ≈ (1/N) ∑_i r_i")
    print("  • Unbiased advantage estimation under representative sampling")
    print("  • Policy gradient theorem ensures convergence properties")
    print("  • Lower variance than REINFORCE with constant baseline")
    print("=" * 80)


def print_interactive_instructions():
    """Print clear instructions for human evaluators"""
    print(f"\n" + "=" * 70)
    print("INTERACTIVE TRAINING GUIDELINES")
    print("=" * 70)
    print("Your Role:")
    print("  • Evaluate model responses for quality and helpfulness")
    print("  • Provide ratings that guide model learning")
    print("  • Focus on practical value and encouragement")
    print()
    print("Rating Criteria:")
    print("  5 = Excellent: Comprehensive, practical, inspiring")
    print("  4 = Good: Helpful, detailed, encouraging")
    print("  3 = Average: Adequate information provided")
    print("  2 = Below Average: Somewhat helpful")
    print("  1 = Poor: Unhelpful, brief, discouraging")
    print()
    print("What to Look For:")
    print("  • Detailed step-by-step instructions")
    print("  • Scientific explanations where relevant")
    print("  • Encouraging tone that builds confidence")
    print("  • Practical tips and troubleshooting advice")
    print("  • Comprehensive coverage of the topic")
    print()
    print("Impact of Your Ratings:")
    print("  • Higher ratings → Model learns to generate similar responses")
    print("  • Lower ratings → Model learns to avoid similar patterns")
    print("  • Group mean serves as baseline for relative comparisons")
    print("  • Your feedback directly shapes model behavior")
    print("=" * 70)


def main():
    print("=" * 80)
    print("GROUP RELATIVE POLICY OPTIMIZATION (GRPO) - STANDALONE")
    print("=" * 80)
    print("Interactive learning from human feedback without value functions")
    print("Mathematical basis: Group-mean advantages for policy gradients")
    print("Innovation: Memory-efficient alternative to traditional PPO")
    print("=" * 80)

    # Print theoretical foundation
    print_grpo_theory()

    install_packages()

    print(f"\nInitializing model: {MODEL_NAME}")

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )

    # Configure tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.resize_token_embeddings(len(tokenizer))
    monitor_memory()

    # Print interactive instructions
    print_interactive_instructions()

    # Evaluate base model
    print("\n" + "=" * 50)
    print("EVALUATING BASE MODEL")
    print("=" * 50)
    base_results = evaluate_stage(model, tokenizer, "BASE")
    save_results_json(base_results, "grpo_base_results.json")

    # Interactive GRPO Training
    print(f"\n{'='*80}")
    print("INTERACTIVE GRPO TRAINING SESSION")
    print(f"{'='*80}")
    print("Process Overview:")
    print("  1. Model generates multiple responses to cooking challenges")
    print("  2. You evaluate each response for quality and helpfulness")
    print("  3. GRPO calculates advantages relative to group mean")
    print("  4. Policy updated based on your preferences")
    print("  5. Model learns to prefer responses similar to your high ratings")
    print("=" * 80)

    print("\nStarting Interactive GRPO Training")
    print("You'll rate model responses to teach it your preferences.")
    print("Press Ctrl+C anytime to skip this step.")

    try:
        model, tokenizer = run_grpo_training(model, tokenizer)
    except KeyboardInterrupt:
        print("\nGRPO training skipped - continuing to final evaluation")

    # Evaluate trained model
    print("\n" + "=" * 50)
    print("EVALUATING TRAINED MODEL")
    print("=" * 50)
    trained_results = evaluate_stage(model, tokenizer, "GRPO")
    save_results_json(trained_results, "grpo_trained_results.json")

    # Save final model
    print(f"\nSaving GRPO-trained model...")
    os.makedirs("./models/grpo_standalone", exist_ok=True)
    model.save_pretrained("./models/grpo_standalone")
    tokenizer.save_pretrained("./models/grpo_standalone")

    # Compare results
    print(f"\n{'='*80}")
    print("COMPARATIVE ANALYSIS: Base vs GRPO")
    print(f"{'='*80}")
    print("Focus: How interactive human feedback improves responses")
    print("Key aspects: Group-mean baselines, direct preference learning")
    print(f"{'='*80}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 60)
        print(f"\n[BASE MODEL (No interactive learning)]:")
        print(f"{base_results[question]}")
        print(f"\n[GRPO MODEL (Interactive human feedback)]:")
        print(f"{trained_results[question]}")
        print("=" * 60)

    # Final analysis
    print(f"\n{'='*80}")
    print("GRPO TRAINING ANALYSIS")
    print(f"{'='*80}")
    print("Interactive Learning Results:")
    print("  • Human feedback integrated: Direct preference learning")
    print("  • Policy adaptation: Model aligned with your quality standards")
    print("  • Memory efficiency: No value function training required")
    print("  • Group baselines: Unbiased advantage estimation")
    print()
    print("GRPO Advantages Demonstrated:")
    print("  • Group mean baseline eliminates critic network complexity")
    print("  • Interactive feedback provides real-time quality signals")
    print("  • Policy gradients directly optimize for human preferences")
    print("  • Memory efficient single-model architecture")
    print()
    print("Mathematical Innovation:")
    print("  • Advantage estimation: A_i = r_i - (1/N)∑r_j")
    print("  • No value function approximation needed")
    print("  • Group sampling for baseline calculation")
    print("  • Direct policy gradient optimization")
    print(f"{'='*80}")

    print(f"\n{'='*80}")
    print("GRPO TRAINING COMPLETED")
    print(f"{'='*80}")
    print("Results saved to:")
    print("  • ./models/grpo_standalone/ - Trained model")
    print("  • ./results/grpo_base_results.json - Base evaluation")
    print("  • ./results/grpo_trained_results.json - Trained evaluation")
    print()
    print("Key Innovation:")
    print("  GRPO demonstrates effective interactive learning from human feedback")
    print("  without requiring value function approximation or critic networks")
    print(f"{'='*80}")

In [4]:
# Execute main
if __name__ == "__main__":
    main()

GROUP RELATIVE POLICY OPTIMIZATION (GRPO) - STANDALONE
Interactive learning from human feedback without value functions
Mathematical basis: Group-mean advantages for policy gradients
Innovation: Memory-efficient alternative to traditional PPO

GRPO THEORETICAL FOUNDATION
Mathematical Innovation: Group-relative advantages without value functions

Core Algorithm:
  Traditional PPO: ∇_θ J = E[∇_θ log π_θ(a|s) × (r - V_φ(s))]
  GRPO Innovation: ∇_θ J = E[∇_θ log π_θ(a|s) × (r_i - (1/N)∑r_j)]
  Key difference: Group mean baseline vs learned value function

Algorithmic Process:
  1. Generate group responses: {y_1, y_2, ..., y_N} ~ π_θ(·|x)
  2. Collect human ratings: r_i = human_evaluation(x, y_i)
  3. Calculate baseline: b = (1/N) ∑_{i=1}^N r_i
  4. Compute advantages: A_i = r_i - b
  5. Policy update: θ ← θ + α ∑_i A_i ∇_θ log π_θ(y_i|x)

Theoretical Advantages:
  • Memory Efficiency: ~50% reduction vs PPO (no critic network)
  • Training Stability: Group mean provides unbiased baseline
  